# BrandSphere AI — Week 1: EDA & Data Understanding
**CRS AI Capstone 2025–26 | Scenario 1**

This notebook covers:
1. Dataset loading and inspection
2. Data quality checks
3. Feature distributions
4. Startup persona analysis
5. Slogan corpus exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette('husl')
print('Libraries loaded ✅')

## 1. Load Datasets

In [ ]:
# Marketing dataset — 200K rows
df_mkt = pd.read_csv('../datasets/raw/marketing_campaign_dataset.csv')
print(f'Marketing: {df_mkt.shape}')
df_mkt.head(3)

In [ ]:
# Startups dataset — 42K rows
df_s = pd.read_csv('../datasets/raw/startups.csv')
print(f'Startups: {df_s.shape}')
df_s.head(3)

## 2. Data Quality Checks

In [ ]:
print('=== MARKETING NULL COUNTS ===')
print(df_mkt.isnull().sum())
print(f'\nDuplicates: {df_mkt.duplicated().sum()}')

In [ ]:
print('=== STARTUPS NULL COUNTS ===')
print(df_s.isnull().sum())
print(f'\nDuplicates: {df_s.duplicated().sum()}')

## 3. Marketing Dataset — Feature Distributions

In [ ]:
# Parse acquisition cost
df_mkt['Cost_Num'] = df_mkt['Acquisition_Cost'].str.replace(r'[$,]','',regex=True).astype(float)
df_mkt['Duration_Days'] = df_mkt['Duration'].str.extract(r'(\d+)').astype(float)
df_mkt['CTR'] = (df_mkt['Clicks'] / df_mkt['Impressions'].clip(lower=1) * 100).round(4)
df_mkt.describe()[['ROI','Engagement_Score','Conversion_Rate','CTR']]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Marketing Dataset — Feature Distributions', fontsize=14, color='white')

for ax, (col, label) in zip(axes.flat, [
    ('ROI', 'ROI Distribution'),
    ('Engagement_Score', 'Engagement Score'),
    ('Conversion_Rate', 'Conversion Rate'),
    ('Campaign_Type', 'Campaign Types'),
    ('Channel_Used', 'Channels Used'),
    ('Location', 'Top Locations'),
]):
    if df_mkt[col].dtype == 'object':
        df_mkt[col].value_counts().head(6).plot(kind='bar', ax=ax, color='#C9A84C')
    else:
        ax.hist(df_mkt[col].dropna(), bins=30, color='#C9A84C', alpha=0.8)
    ax.set_title(label, color='white', fontsize=10)
    ax.tick_params(colors='gray')

plt.tight_layout()
plt.savefig('../assets/sample_exports/eda_marketing_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Chart saved')

## 4. Startup Persona Analysis

In [ ]:
# Infer industries from description keywords
INDUSTRY_KEYWORDS = {
    'Technology': ['software','saas','app','tech','ai','data','cloud','platform'],
    'Healthcare': ['health','medical','care','wellness','therapy','patient'],
    'Finance': ['finance','fintech','payment','invest','bank','credit'],
    'Education': ['education','learning','course','student','school'],
    'Food': ['food','restaurant','drink','beverage','meal','recipe'],
}

def infer_industry(text):
    text = str(text).lower()
    for ind, kws in INDUSTRY_KEYWORDS.items():
        if any(k in text for k in kws):
            return ind
    return 'Other'

df_s['industry'] = df_s['description'].apply(infer_industry)
print(df_s['industry'].value_counts())

df_s['industry'].value_counts().plot(kind='bar', color='#C9A84C', figsize=(8,4))
plt.title('Startup Industry Distribution', color='white')
plt.tight_layout()
plt.savefig('../assets/sample_exports/eda_startup_industries.png', dpi=100)
plt.show()

## 5. Slogan / Tagline Exploration

In [ ]:
df_s_clean = df_s.dropna(subset=['tagline'])
df_s_clean['word_count'] = df_s_clean['tagline'].str.split().str.len()
df_s_clean['char_count'] = df_s_clean['tagline'].str.len()

print(f'Taglines available: {len(df_s_clean):,}')
print(f'Avg word count: {df_s_clean["word_count"].mean():.1f}')
print(f'Avg char count: {df_s_clean["char_count"].mean():.1f}')

df_s_clean['word_count'].value_counts().sort_index().plot(kind='bar', color='#C9A84C', figsize=(10,4))
plt.title('Tagline Word Count Distribution', color='white')
plt.xlabel('Words')
plt.tight_layout()
plt.show()

In [ ]:
# Most common words in startup taglines (NLTK preprocessing)
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
from collections import Counter
import re

stop = set(stopwords.words('english'))
all_words = []
for tag in df_s_clean['tagline'].dropna():
    words = re.sub(r'[^a-z\s]', '', tag.lower()).split()
    all_words.extend([w for w in words if w not in stop and len(w) > 2])

top_words = Counter(all_words).most_common(20)
words, counts = zip(*top_words)
plt.figure(figsize=(12, 4))
plt.bar(words, counts, color='#C9A84C')
plt.title('Top 20 Words in Startup Taglines', color='white')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print('Top words:', words[:10])

## Summary
- Marketing dataset: 200K rows, synthetic outcomes (ROI, Engagement are uniformly distributed — noted for ML evaluation)
- Startups: 42K startups, majority in Technology sector
- Taglines: average 5–7 words, strong action verbs dominate